# Lenta YOLO Baseline (Kaggle Notebook)

2-stage pipeline for price-tag detection:
1. **GT Dataset Builder** — converts CSV+video → YOLO-format labels
2. **Fine-tuning** — trains `yolov9t` on labeled frames (~5-10 min on Kaggle GPU)
3. **Inference** — YOLO detects price-tag bboxes frame-by-frame

Pipeline:
`video → frames → rotate → YOLO detect → crops → debug CSV → metrics → zip`


In [1]:
#Для загрузки датасета из гугла
!pip install -q gdown
!gdown --folder "https://drive.google.com/drive/folders/1XRrRB7y66RU4lxZiH7a6H_b8fOvKgOQl" -O /kaggle/working

zsh:1: command not found: pip
zsh:1: command not found: gdown


In [2]:
!pip install -q ultralytics

zsh:1: command not found: pip


## 1. Imports and configuration


In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import re
import shutil
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ----------------------------
# User-editable configuration
# ----------------------------
# RUN_MODE = "single": process only INPUT_VIDEO_PATH
# RUN_MODE = "batch" : auto-discover all *.mp4 under DATA_ROOT
RUN_MODE = "batch"

DATA_ROOT = "/kaggle/working"
CASE_FILTER = None          # e.g. "43_15" to filter in batch mode
INCLUDE_UNLABELED = True

# Single mode paths (used when RUN_MODE="single")
INPUT_VIDEO_PATH = f"{DATA_ROOT}/43_15/43_15.mp4"
GT_ANNOTATIONS_PATH = f"{DATA_ROOT}/43_15/43_15.csv"  # or None

# Output
OUTPUT_DIR = "/kaggle/working/baseline_yolo_candidates_single"
BATCH_OUTPUT_ROOT = "/kaggle/working/baseline_yolo_candidates_batch"

SAMPLE_FPS = 2.0          # upper bound for adaptive sampling when motion is high
MIN_SAMPLE_FPS = 1.0      # lower bound when robot is nearly static
FLOW_MAGNITUDE_THR = 0.15 # mean Farneback flow magnitude threshold in downsampled pixels
FLOW_RESIZE_WIDTH = 320   # downsample width for optical flow speed/robustness
MAX_FRAMES = None
PADDING = 0.05           # padding applied to each detected bbox before cropping
RANDOM_SEED = 42

# Robot videos are sideways — fixed CCW rotation.
# "none" | "cw" | "ccw" | "180"
FRAME_ROTATION_MODE = "ccw"

SAVE_PREPROCESSED_FRAMES = True
PREPROCESS_PREVIEW_N = 8
TOP_N_DEBUG_FRAMES = 6
TOP_N_CROPS = 12

# Temporal top-K: best frames per time window (before YOLO)
TEMPORAL_WINDOW_SEC = 3.0
TOP_K_FRAMES_PER_WINDOW = 3
SHOW_TEMPORAL_TOPK_DEBUG = True
TEMPORAL_TOPK_DEBUG_MAX_WINDOWS = None  # None = show all windows

# Fast frame dedup inside each temporal window (before YOLO)
FRAME_SIMILARITY_THR = 0.93       # 0.90–0.95: skip if cosine similarity >= thr
FRAME_SIMILARITY_SIZE = 64        # downsample side for signature (speed)

# Bbox tracking across frames (after YOLO): Hungarian + combined cost
TRACK_COST_ALPHA = 0.7            # weight for (1 - IoU)
TRACK_COST_BETA = 0.3             # weight for normalized center distance
TRACK_MAX_MATCH_COST = 0.55       # reject assignment if cost above this
TRACK_MAX_GAP_MS = 2000           # link detections across gaps (not tied to 3s window)

# HARD FRAME FILTERS (disabled): glare / blur / brightness rejection before YOLO
# MAX_GLARE_RATIO_FOR_OCR = 0.08
# TENENGRAD_MEDIAN_FACTOR = 0.5
# MIN_BRIGHTNESS_FOR_OCR = 40.0
# MAX_BRIGHTNESS_FOR_OCR = 220.0

# Frame quality ROI (after rotation): shelf / price-tag zone
SCORE_ROI_Y_START_FRAC = 0.2
SCORE_ROI_X_START_FRAC = 0.15
SCORE_ROI_X_END_FRAC = 0.85
SCORE_PERCENTILE_LOW = 10.0
SCORE_PERCENTILE_HIGH = 90.0

# ----------------------------
# YOLO configuration
# ----------------------------
YOLO_BASE_MODEL = "yolov9c.pt"   # классическая полная модель (~25M params)  # downloaded automatically by ultralytics
YOLO_FINETUNED_MODEL = None     # filled after fine-tuning, or set manually

YOLO_CONF_THR = 0.25
YOLO_IOU_NMS = 0.45
YOLO_IMG_SIZE = 640

YOLO_MIN_AREA_RATIO = 0.0005
YOLO_MAX_AREA_RATIO = 0.30
YOLO_ASPECT_RATIO_MIN = 0.15
YOLO_ASPECT_RATIO_MAX = 10.0

# ----------------------------
# Fine-tune configuration
# ----------------------------
ENABLE_FINETUNE = True       # False = use base/pre-specified model as-is
FINETUNE_EPOCHS = 50
FINETUNE_BATCH = 8               # эффективный batch = 8*n_gpus при DDP
FINETUNE_PATIENCE = 10
FINETUNE_IMG_SIZE = 640
FINETUNE_DATA_DIR = "/kaggle/working/finetune_dataset"
FINETUNE_RUN_DIR = "/kaggle/working/yolo_runs"
FINETUNE_SPLIT_RATIO = 0.8   # fraction of timestamps used for YOLO internal train/val

# ----------------------------
# Train / test video split
# ----------------------------
# Labeled videos are split into:
#   finetune set  — used to train the model (never evaluated as test)
#   test set      — held out, used only for final evaluation
#
# Option A (auto): last FINETUNE_HOLDOUT_RATIO fraction of labeled videos → test
# Option B (manual): list exact video stems in TEST_VIDEO_NAMES
FINETUNE_HOLDOUT_RATIO = 0.2   # fraction of labeled videos held out for test
TEST_VIDEO_NAMES: list[str] | None = None  # e.g. ["26_12-20", "43_15"]; None = auto
# GPU авто-детект при старте ноутбука
import torch as _torch
_n_gpus = _torch.cuda.device_count()
TRAIN_DEVICE = list(range(_n_gpus)) if _n_gpus > 1 else (0 if _n_gpus == 1 else "cpu")
INFER_DEVICE: int | str = 0 if _n_gpus >= 1 else "cpu"

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["axes.grid"] = False


ModuleNotFoundError: No module named 'cv2'

## 2. Utility dataclasses


In [ ]:
@dataclass
class Candidate:
    bbox: tuple[int, int, int, int]  # x_min, y_min, x_max, y_max
    score: float
    source: str
    matched_texts: list[str]


## 3. Video inspection


In [ ]:
def inspect_video(video_path: str) -> dict[str, float | int | str]:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    cap.release()

    if fps <= 0:
        fps = 25.0

    duration_sec = frame_count / fps if fps > 0 else 0.0
    return {
        "video_path": video_path,
        "fps": fps,
        "frame_count": frame_count,
        "width": width,
        "height": height,
        "duration_sec": duration_sec,
    }


## 4. Frame extraction


In [ ]:
def extract_frames(
    video_path: str,
    frames_dir: str | Path,
    sample_fps: float = 2.0,
    min_sample_fps: float = 1.0,
    flow_magnitude_thr: float = 0.15,
    flow_resize_width: int = 320,
    max_frames: int | None = None,
) -> list[dict[str, Any]]:
    frames_dir = Path(frames_dir)
    frames_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if fps <= 0:
        fps = 25.0

    max_sample_fps = max(float(sample_fps), 1e-6)
    min_sample_fps = max(float(min_sample_fps), 1e-6)
    if min_sample_fps > max_sample_fps:
        min_sample_fps = max_sample_fps

    max_frame_step = max(1, int(round(fps / max_sample_fps)))
    min_frame_step = max(1, int(round(fps / min_sample_fps)))
    flow_magnitude_thr = max(float(flow_magnitude_thr), 1e-6)
    flow_resize_width = max(int(flow_resize_width), 64)

    records: list[dict[str, Any]] = []
    frame_index = 0
    saved = 0
    next_save_at = 0
    prev_small_gray: np.ndarray | None = None

    pbar_total = frame_count if frame_count > 0 else None
    with tqdm(total=pbar_total, desc="Extracting frames (adaptive flow)") as pbar:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            h, w = frame.shape[:2]
            if w > 0:
                scale = min(1.0, float(flow_resize_width) / float(w))
            else:
                scale = 1.0
            target_w = max(16, int(round(w * scale)))
            target_h = max(16, int(round(h * scale)))
            small = cv2.resize(frame, (target_w, target_h), interpolation=cv2.INTER_AREA)
            gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

            flow_mean_mag = 0.0
            motion_ratio = 0.0
            if prev_small_gray is not None and prev_small_gray.shape == gray.shape:
                flow = cv2.calcOpticalFlowFarneback(
                    prev_small_gray,
                    gray,
                    None,
                    pyr_scale=0.5,
                    levels=3,
                    winsize=15,
                    iterations=3,
                    poly_n=5,
                    poly_sigma=1.2,
                    flags=0,
                )
                flow_mag = np.sqrt(flow[..., 0] ** 2 + flow[..., 1] ** 2)
                flow_mean_mag = float(np.mean(flow_mag))
                motion_ratio = float(np.clip(flow_mean_mag / flow_magnitude_thr, 0.0, 1.0))

            adaptive_step = int(
                round(min_frame_step - motion_ratio * float(min_frame_step - max_frame_step))
            )
            adaptive_step = int(np.clip(adaptive_step, max_frame_step, min_frame_step))

            if frame_index >= next_save_at:
                timestamp_ms = int(round((frame_index / fps) * 1000.0))
                frame_name = f"frame_{frame_index:08d}_{timestamp_ms:010d}ms.jpg"
                frame_path = frames_dir / frame_name
                cv2.imwrite(str(frame_path), frame)

                records.append(
                    {
                        "frame_index": frame_index,
                        "timestamp_ms": timestamp_ms,
                        "frame_path": str(frame_path),
                        "flow_mean_mag": float(flow_mean_mag),
                        "motion_ratio": float(motion_ratio),
                        "adaptive_step": int(adaptive_step),
                    }
                )
                saved += 1
                next_save_at = frame_index + adaptive_step
                if max_frames is not None and saved >= max_frames:
                    break

            prev_small_gray = gray
            frame_index += 1
            if pbar_total is not None:
                pbar.update(1)

    cap.release()
    return records


## 5. Image quality functions


In [ ]:
def _score_roi_gray(gray: np.ndarray) -> np.ndarray:
    h, w = gray.shape[:2]
    if h <= 0 or w <= 0:
        return gray
    y1 = int(np.clip(h * SCORE_ROI_Y_START_FRAC, 0, h - 1))
    x1 = int(np.clip(w * SCORE_ROI_X_START_FRAC, 0, w - 1))
    x2 = int(np.clip(w * SCORE_ROI_X_END_FRAC, x1 + 1, w))
    return gray[y1:h, x1:x2]


def compute_tenengrad_roi(image: np.ndarray, use_roi: bool = True) -> float:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    roi = _score_roi_gray(gray) if use_roi else gray
    if roi.size == 0:
        return 0.0
    gx = cv2.Sobel(roi, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(roi, cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = np.sqrt(gx * gx + gy * gy)
    return float(np.mean(grad_mag))


def compute_sharpness(image: np.ndarray) -> float:
    return compute_tenengrad_roi(image, use_roi=True)


def compute_brightness(image: np.ndarray, use_roi: bool = True) -> float:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    roi = _score_roi_gray(gray) if use_roi else gray
    if roi.size == 0:
        return 0.0
    return float(np.mean(roi))


def compute_contrast(image: np.ndarray, use_roi: bool = True) -> float:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    roi = _score_roi_gray(gray) if use_roi else gray
    if roi.size == 0:
        return 0.0
    return float(np.std(roi))


def compute_glare_ratio(
    image: np.ndarray,
    threshold: int = 60,
    kernel_size: int = 15,
    use_roi: bool = True,
) -> float:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    roi = _score_roi_gray(gray) if use_roi else gray
    if roi.size == 0:
        return 0.0
    kernel = np.ones((kernel_size, kernel_size), dtype=np.uint8)
    tophat = cv2.morphologyEx(roi, cv2.MORPH_TOPHAT, kernel)
    return float(np.mean(tophat > threshold))


def _frame_similarity_signature(image_bgr: np.ndarray, size: int = FRAME_SIMILARITY_SIZE) -> np.ndarray:
    """Normalized gray thumbnail vector on ROI — for fast cosine similarity."""
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY) if image_bgr.ndim == 3 else image_bgr
    roi = _score_roi_gray(gray)
    if roi.size == 0:
        roi = gray
    side = max(16, int(size))
    thumb = cv2.resize(roi, (side, side), interpolation=cv2.INTER_AREA)
    vec = thumb.astype(np.float32).reshape(-1)
    vec -= float(vec.mean())
    norm = float(np.linalg.norm(vec))
    if norm > 1e-6:
        vec /= norm
    return vec


def compute_frame_similarity(
    image_a: np.ndarray,
    image_b: np.ndarray,
    size: int = FRAME_SIMILARITY_SIZE,
) -> float:
    """Cosine similarity in [0, 1] on downsampled ROI (1.0 = nearly identical)."""
    va = _frame_similarity_signature(image_a, size=size)
    vb = _frame_similarity_signature(image_b, size=size)
    return float(np.clip(float(np.dot(va, vb)), 0.0, 1.0))


def compute_frame_raw_metrics(image_bgr: np.ndarray) -> dict[str, float]:
    return {
        "sharpness": compute_tenengrad_roi(image_bgr, use_roi=True),
        "brightness": compute_brightness(image_bgr, use_roi=True),
        "contrast": compute_contrast(image_bgr, use_roi=True),
        "glare_ratio": compute_glare_ratio(image_bgr, threshold=60, kernel_size=15, use_roi=True),
    }


def build_video_percentile_stats(
    raw_metrics_list: list[dict[str, float]],
    p_low: float = SCORE_PERCENTILE_LOW,
    p_high: float = SCORE_PERCENTILE_HIGH,
) -> dict[str, tuple[float, float]]:
    keys = ("sharpness", "contrast", "brightness", "glare_ratio")
    stats: dict[str, tuple[float, float]] = {}
    for key in keys:
        vals = [float(m[key]) for m in raw_metrics_list if key in m]
        if not vals:
            stats[key] = (0.0, 1.0)
            continue
        if len(vals) == 1:
            v = vals[0]
            stats[key] = (v, v + 1e-6)
            continue
        p10, p90 = np.percentile(vals, [p_low, p_high])
        if float(p90) <= float(p10):
            p90 = float(p10) + 1e-6
        stats[key] = (float(p10), float(p90))
    return stats


def _percentile_norm(value: float, p10: float, p90: float) -> float:
    return float(np.clip((float(value) - float(p10)) / max(float(p90) - float(p10), 1e-6), 0.0, 1.0))


def raw_metrics_to_frame_scores(
    raw: dict[str, float],
    percentile_stats: dict[str, tuple[float, float]],
) -> dict[str, float]:
    sharpness_norm = _percentile_norm(raw["sharpness"], *percentile_stats["sharpness"])
    contrast_norm = _percentile_norm(raw["contrast"], *percentile_stats["contrast"])
    brightness_norm = _percentile_norm(raw["brightness"], *percentile_stats["brightness"])
    glare_norm = _percentile_norm(raw["glare_ratio"], *percentile_stats["glare_ratio"])

    custom_frame_score = (
        0.45 * sharpness_norm
        + 0.15 * contrast_norm
        + 0.10 * brightness_norm
        + 0.30 * (1.0 - glare_norm)
    )

    return {
        "sharpness": float(raw["sharpness"]),
        "brightness": float(raw["brightness"]),
        "contrast": float(raw["contrast"]),
        "glare_ratio": float(raw["glare_ratio"]),
        "sharpness_score": float(sharpness_norm),
        "custom_frame_score": float(custom_frame_score),
    }


def preprocess_frame_bundle(image_bgr: np.ndarray) -> dict[str, np.ndarray]:
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    # Local contrast normalization is useful for shelf reflections.
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(gray)
    denoised = cv2.bilateralFilter(clahe, 7, 55, 55)
    blur = cv2.GaussianBlur(denoised, (0, 0), 1.2)
    sharpen = cv2.addWeighted(denoised, 1.55, blur, -0.55, 0)

    return {
        "gray": gray,
        "clahe": clahe,
        "denoised": denoised,
        "sharpen": sharpen,
        "ocr_ready_bgr": cv2.cvtColor(sharpen, cv2.COLOR_GRAY2BGR),
    }


def compute_frame_scores(
    image_bgr: np.ndarray,
    percentile_stats: dict[str, tuple[float, float]] | None = None,
) -> dict[str, float]:
    raw = compute_frame_raw_metrics(image_bgr)
    if percentile_stats is None:
        percentile_stats = build_video_percentile_stats([raw])
    return raw_metrics_to_frame_scores(raw, percentile_stats)


def compute_crop_quality(crop: np.ndarray, candidate_score: float) -> dict[str, float]:
    if crop is None or crop.size == 0:
        return {
            "sharpness": 0.0,
            "brightness": 0.0,
            "contrast": 0.0,
            "quality_score": 0.0,
        }

    sharpness = compute_tenengrad_roi(crop, use_roi=False)
    brightness = compute_brightness(crop, use_roi=False)
    contrast = compute_contrast(crop, use_roi=False)

    sharpness_norm = float(np.clip(sharpness / 500.0, 0.0, 1.0))
    contrast_norm = float(np.clip(contrast / 80.0, 0.0, 1.0))
    brightness_norm = float(np.clip(1.0 - abs(brightness - 127.0) / 127.0, 0.0, 1.0))

    quality_score = (
        0.5 * sharpness_norm
        + 0.2 * contrast_norm
        + 0.2 * brightness_norm
        + 0.1 * float(np.clip(candidate_score, 0.0, 2.0) / 2.0)
    )

    return {
        "sharpness": sharpness,
        "brightness": brightness,
        "contrast": contrast,
        "quality_score": float(quality_score),
    }


## 6. GT Dataset Builder


In [ ]:
def _to_float_series_local(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str)
        .str.replace(" ", " ", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def _apply_rotation_local(image: np.ndarray, rotation: str) -> np.ndarray:
    rot = (rotation or "none").lower()
    if rot == "cw":
        return cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    if rot == "ccw":
        return cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
    if rot == "180":
        return cv2.rotate(image, cv2.ROTATE_180)
    return image


def _rotate_bbox_coords(
    x1: float, y1: float, x2: float, y2: float,
    orig_w: int, orig_h: int, rotation: str,
) -> tuple[float, float, float, float]:
    rot = (rotation or "none").lower()
    if rot == "none":
        return x1, y1, x2, y2
    corners = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
    rotated = []
    for x, y in corners:
        if rot == "cw":
            rotated.append((orig_h - 1.0 - y, x))
        elif rot == "ccw":
            rotated.append((y, orig_w - 1.0 - x))
        elif rot == "180":
            rotated.append((orig_w - 1.0 - x, orig_h - 1.0 - y))
    xs = [p[0] for p in rotated]
    ys = [p[1] for p in rotated]
    return min(xs), min(ys), max(xs), max(ys)


def _bbox_to_yolo(
    x1: float, y1: float, x2: float, y2: float, img_w: int, img_h: int
) -> tuple[float, float, float, float] | None:
    cx = (x1 + x2) / 2.0 / img_w
    cy = (y1 + y2) / 2.0 / img_h
    w = (x2 - x1) / img_w
    h = (y2 - y1) / img_h
    if w <= 0 or h <= 0:
        return None
    return (
        float(np.clip(cx, 0.0, 1.0)),
        float(np.clip(cy, 0.0, 1.0)),
        float(np.clip(w, 0.0, 1.0)),
        float(np.clip(h, 0.0, 1.0)),
    )


def _seek_frame(cap: cv2.VideoCapture, timestamp_ms: int, fps: float) -> np.ndarray | None:
    frame_idx = int(round(timestamp_ms / 1000.0 * fps))
    for delta in [0, -1, 1, -2, 2, -5, 5]:
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, frame_idx + delta))
        ok, frame = cap.read()
        if ok:
            return frame
    return None


def build_yolo_dataset(
    gt_csv_paths: list[str],
    video_paths: list[str],
    out_dir: str,
    rotation: str = "ccw",
    split_ratio: float = 0.8,
) -> str:
    out = Path(out_dir)
    for split in ("train", "val"):
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)

    all_samples: list[dict[str, Any]] = []

    for csv_path, video_path in zip(gt_csv_paths, video_paths):
        gt = pd.read_csv(csv_path, sep=None, engine="python")
        for col in ["frame_timestamp", "x_min", "y_min", "x_max", "y_max"]:
            gt[col] = _to_float_series_local(gt[col])
        gt = gt.dropna(subset=["frame_timestamp", "x_min", "y_min", "x_max", "y_max"])

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"[WARN] Cannot open video: {video_path}")
            continue
        fps = float(cap.get(cv2.CAP_PROP_FPS) or 25.0)
        orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        by_ts = gt.groupby("frame_timestamp")
        for ts in sorted(by_ts.groups.keys()):
            frame = _seek_frame(cap, int(ts), fps)
            if frame is None:
                continue
            frame_rot = _apply_rotation_local(frame, rotation)
            rot_h, rot_w = frame_rot.shape[:2]

            labels: list[str] = []
            for row in by_ts.get_group(ts).itertuples(index=False):
                rx1, ry1, rx2, ry2 = _rotate_bbox_coords(
                    float(row.x_min), float(row.y_min),
                    float(row.x_max), float(row.y_max),
                    orig_w, orig_h, rotation,
                )
                yolo = _bbox_to_yolo(rx1, ry1, rx2, ry2, rot_w, rot_h)
                if yolo is not None:
                    labels.append(f"0 {yolo[0]:.6f} {yolo[1]:.6f} {yolo[2]:.6f} {yolo[3]:.6f}")

            if labels:
                all_samples.append({
                    "ts": int(ts),
                    "frame": frame_rot,
                    "labels": labels,
                    "stem": f"{Path(video_path).stem}_{int(ts):010d}",
                })

        cap.release()

    if not all_samples:
        raise RuntimeError("No samples extracted from GT CSV + video pairs.")

    all_samples.sort(key=lambda s: s["ts"])
    n_train = max(1, int(len(all_samples) * split_ratio))
    splits = ["train"] * n_train + ["val"] * max(1, len(all_samples) - n_train)

    for sample, split in zip(all_samples, splits):
        stem = sample["stem"]
        cv2.imwrite(str(out / "images" / split / f"{stem}.jpg"), sample["frame"])
        (out / "labels" / split / f"{stem}.txt").write_text("\n".join(sample["labels"]))

    data_yaml_content = (
        f"path: {str(out)}\n"
        "train: images/train\n"
        "val: images/val\n"
        "nc: 1\n"
        "names:\n"
        "  - price_tag\n"
    )
    dataset_yaml = out / "dataset.yaml"
    dataset_yaml.write_text(data_yaml_content)
    print(f"Dataset built: {splits.count('train')} train / {splits.count('val')} val → {out}")
    return str(dataset_yaml)


## 7. Fine-tuning


In [ ]:
def run_finetune(
    data_yaml: str,
    base_model: str,
    epochs: int,
    batch: int,
    img_size: int,
    patience: int,
    run_dir: str,
    train_device=None,
    workers: int = 4,
) -> str:
    from ultralytics import YOLO
    model = YOLO(base_model)
    model.train(
        data=data_yaml,
        epochs=epochs,
        batch=batch,
        imgsz=img_size,
        patience=patience,
        project=run_dir,
        name="price_tag_ft",
        exist_ok=True,
        verbose=True,
        plots=True,
        device=train_device,
        workers=workers,
    )
    best = Path(run_dir) / "price_tag_ft" / "weights" / "best.pt"
    if best.exists():
        print(f"Fine-tuning done. Best weights: {best}")
        return str(best)
    last = Path(run_dir) / "price_tag_ft" / "weights" / "last.pt"
    print(f"[WARN] best.pt not found, falling back to last.pt: {last}")
    return str(last)


In [ ]:
def show_train_batches(run_dir: str, name: str = "price_tag_ft", max_batches: int = 3) -> None:
    """Display train_batch*.jpg images that ultralytics generates at the start of training."""
    import matplotlib.pyplot as plt

    batch_dir = Path(run_dir) / name
    batch_imgs = sorted(batch_dir.glob("train_batch*.jpg"))[:max_batches]
    if not batch_imgs:
        print(f"[VIZ] No train_batch*.jpg found in {batch_dir}")
        return
    fig, axes = plt.subplots(1, len(batch_imgs), figsize=(10 * len(batch_imgs), 8))
    if len(batch_imgs) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, batch_imgs):
        img_rgb = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.set_title(img_path.name, fontsize=10)
        ax.axis("off")
    plt.suptitle("Finetune GT batches (разметка на обучающих данных)", fontsize=13)
    plt.tight_layout()
    plt.show()
    print(f"[VIZ] Showed {len(batch_imgs)} batch preview(s) from {batch_dir}")


def show_inference_samples(
    result_df: "pd.DataFrame",
    video_path: str,
    case_id: str,
    n_samples: int = 8,
    rotation: str = "ccw",
) -> None:
    """Show N sample frames from result_df with detected bboxes drawn."""
    import matplotlib.pyplot as plt

    if result_df is None or result_df.empty:
        print(f"[VIZ] No detections for {case_id}")
        return

    step = max(1, len(result_df) // n_samples)
    rows = result_df.iloc[::step][:n_samples]

    rot_map = {
        "cw": cv2.ROTATE_90_CLOCKWISE,
        "ccw": cv2.ROTATE_90_COUNTERCLOCKWISE,
        "180": cv2.ROTATE_180,
    }
    rot_code = rot_map.get((rotation or "none").lower())

    cap = cv2.VideoCapture(video_path)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 25.0)

    frames_drawn = []
    for _, row in rows.iterrows():
        ts = int(float(row["timestamp_ms"]))
        frame_idx = int(round(ts / 1000.0 * fps))
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
        if not ok:
            continue
        if rot_code is not None:
            frame = cv2.rotate(frame, rot_code)

        x1 = int(float(row.get("x_min", 0)))
        y1 = int(float(row.get("y_min", 0)))
        x2 = int(float(row.get("x_max", 0)))
        y2 = int(float(row.get("y_max", 0)))
        score = float(row.get("candidate_score", 0.0))
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 80, 0), 3)
        cv2.putText(frame, f"{score:.2f}", (x1, max(0, y1 - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 80, 0), 2)
        frames_drawn.append((frame, ts))

    cap.release()
    if not frames_drawn:
        print(f"[VIZ] Could not read frames for {case_id}")
        return

    n = len(frames_drawn)
    ncols = min(4, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes_flat = np.array(axes).reshape(-1) if n > 1 else [axes]

    for ax, (frame, ts) in zip(axes_flat, frames_drawn):
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f"ts={ts}ms", fontsize=8)
        ax.axis("off")
    for ax in axes_flat[n:]:
        ax.axis("off")

    plt.suptitle(f"Model detections — {case_id}", fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f"[VIZ] Showed {n} inference samples for {case_id}")


## 8. YOLO inference


In [ ]:
def box_iou(box_a: tuple[int, int, int, int], box_b: tuple[int, int, int, int]) -> float:
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    inter_x1, inter_y1 = max(ax1, bx1), max(ay1, by1)
    inter_x2, inter_y2 = min(ax2, bx2), min(ay2, by2)
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter_area
    return float(inter_area / union) if union > 0 else 0.0


def nms(candidates: list[Candidate], iou_threshold: float = 0.5) -> list[Candidate]:
    if not candidates:
        return []
    order = np.argsort([c.score for c in candidates])[::-1]
    keep: list[Candidate] = []
    while len(order) > 0:
        idx = int(order[0])
        current = candidates[idx]
        keep.append(current)
        remaining = [int(j) for j in order[1:] if box_iou(current.bbox, candidates[int(j)].bbox) < iou_threshold]
        order = np.array(remaining, dtype=int)
    return keep


def _expand_and_clip_bbox(
    box: tuple[int, int, int, int], image_shape: tuple[int, ...], padding: float
) -> tuple[int, int, int, int]:
    h, w = image_shape[:2]
    x1, y1, x2, y2 = box
    bw, bh = max(1, x2 - x1), max(1, y2 - y1)
    pad_x, pad_y = int(round(bw * padding)), int(round(bh * padding))
    nx1 = max(0, x1 - pad_x)
    ny1 = max(0, y1 - pad_y)
    nx2 = min(w - 1, x2 + pad_x)
    ny2 = min(h - 1, y2 + pad_y)
    if nx2 <= nx1:
        nx2 = min(w - 1, nx1 + 1)
    if ny2 <= ny1:
        ny2 = min(h - 1, ny1 + 1)
    return (nx1, ny1, nx2, ny2)


_yolo_model = None


def get_yolo_model() -> Any:
    global _yolo_model
    if _yolo_model is None:
        from ultralytics import YOLO
        weights = YOLO_FINETUNED_MODEL or YOLO_BASE_MODEL
        _yolo_model = YOLO(weights)
        print(f"[INFO] YOLO model loaded: {weights}")
    return _yolo_model


def detect_with_yolo(image: np.ndarray) -> list[Candidate]:
    model = get_yolo_model()
    h, w = image.shape[:2]
    img_area = float(h * w)

    import torch as _t
    _infer_dev = 0 if _t.cuda.is_available() else "cpu"
    results = model.predict(
        image, conf=YOLO_CONF_THR, iou=YOLO_IOU_NMS,
        imgsz=YOLO_IMG_SIZE, verbose=False, device=_infer_dev,
    )

    candidates: list[Candidate] = []
    for result in results:
        if result.boxes is None or len(result.boxes) == 0:
            continue
        for xyxy, conf_val in zip(result.boxes.xyxy.cpu().numpy(), result.boxes.conf.cpu().numpy()):
            x1, y1, x2, y2 = int(xyxy[0]), int(xyxy[1]), int(xyxy[2]), int(xyxy[3])
            bw, bh = max(1, x2 - x1), max(1, y2 - y1)
            area_ratio = (bw * bh) / img_area
            aspect_ratio = bw / float(max(1, bh))
            if not (YOLO_MIN_AREA_RATIO <= area_ratio <= YOLO_MAX_AREA_RATIO):
                continue
            if not (YOLO_ASPECT_RATIO_MIN <= aspect_ratio <= YOLO_ASPECT_RATIO_MAX):
                continue
            bbox = _expand_and_clip_bbox((x1, y1, x2, y2), image.shape, PADDING)
            candidates.append(Candidate(bbox=bbox, score=float(conf_val), source="yolo", matched_texts=[]))

    return candidates


## 9. Optional fallback without OCR


In [ ]:
def find_rectangular_candidates(
    image: np.ndarray,
    min_area_ratio: float = 0.0005,
    max_area_ratio: float = 0.25,
    aspect_ratio_range: tuple[float, float] = (0.2, 8.0),
    nms_iou: float = 0.4,
) -> list[Candidate]:
    h, w = image.shape[:2]
    img_area = float(h * w)

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    edges = cv2.Canny(blur, 50, 150)
    binary = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        31,
        7,
    )
    mask = cv2.bitwise_or(edges, binary)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    candidates: list[Candidate] = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area <= 0:
            continue

        area_ratio = area / img_area
        if area_ratio < min_area_ratio or area_ratio > max_area_ratio:
            continue

        peri = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.03 * peri, True)

        x, y, bw, bh = cv2.boundingRect(approx if len(approx) >= 4 else cnt)
        if bw <= 2 or bh <= 2:
            continue

        aspect_ratio = bw / float(max(1, bh))
        if aspect_ratio < aspect_ratio_range[0] or aspect_ratio > aspect_ratio_range[1]:
            continue

        rect_fill = area / float(max(1, bw * bh))
        if rect_fill < 0.35:
            continue

        bbox = _expand_and_clip_bbox((x, y, x + bw, y + bh), image.shape, padding=0.05)
        score = 0.25 + 0.5 * min(1.0, rect_fill) + 0.25 * min(1.0, area_ratio / max_area_ratio)

        candidates.append(
            Candidate(
                bbox=bbox,
                score=float(score),
                source="contours_fallback",
                matched_texts=[],
            )
        )

    candidates = sorted(candidates, key=lambda c: c.score, reverse=True)[:60]
    return nms(candidates, iou_threshold=nms_iou)


## 10. Evaluation utilities


In [ ]:
def _to_float_series(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str)
        .str.replace(" ", " ", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def load_gt_annotations(gt_path: str | Path) -> pd.DataFrame:
    gt_path = Path(gt_path)
    if not gt_path.exists():
        raise FileNotFoundError(f"GT file does not exist: {gt_path}")
    gt = pd.read_csv(gt_path, sep=None, engine="python")
    required = ["frame_timestamp", "x_min", "y_min", "x_max", "y_max"]
    missing = [c for c in required if c not in gt.columns]
    if missing:
        raise ValueError(f"GT CSV missing required columns: {missing}")
    gt = gt.copy()
    for col in required:
        gt[col] = _to_float_series(gt[col])
    if "filename" in gt.columns:
        gt["filename"] = gt["filename"].astype(str).str.strip()
    return gt.dropna(subset=required).reset_index(drop=True)


def _map_gt_to_extracted_frames(
    gt_df: pd.DataFrame,
    frame_records: list[dict[str, Any]],
    sample_fps: float,
) -> tuple[pd.DataFrame, int]:
    if gt_df.empty or not frame_records:
        return gt_df.copy(), 0
    frame_ts = np.array([int(r["timestamp_ms"]) for r in frame_records], dtype=np.int64)
    frame_idx = np.array([int(r["frame_index"]) for r in frame_records], dtype=np.int64)

    if len(frame_ts) > 1:
        dt = np.diff(frame_ts)
        dt = dt[dt > 0]
        median_interval_ms = int(round(float(np.median(dt)))) if len(dt) else int(round(1000.0 / max(sample_fps, 1e-6)))
    else:
        median_interval_ms = int(round(1000.0 / max(sample_fps, 1e-6)))

    tolerance_ms = max(1, int(round(0.5 * max(median_interval_ms, 1))))
    mapped_rows = []
    for row in gt_df.itertuples(index=False):
        target = int(round(float(row.frame_timestamp)))
        nearest_idx = int(np.argmin(np.abs(frame_ts - target)))
        nearest_ts = int(frame_ts[nearest_idx])
        if abs(nearest_ts - target) <= tolerance_ms:
            mapped = dict(row._asdict())
            mapped["mapped_frame_index"] = int(frame_idx[nearest_idx])
            mapped["mapped_timestamp_ms"] = nearest_ts
            mapped["timestamp_delta_ms"] = abs(nearest_ts - target)
            mapped_rows.append(mapped)
    return pd.DataFrame(mapped_rows), tolerance_ms


def _compute_ap_from_flags(tp_flags: list[int], fp_flags: list[int], total_gt: int) -> float:
    if total_gt <= 0 or not tp_flags:
        return 0.0
    tp = np.cumsum(np.array(tp_flags, dtype=float))
    fp_cum = np.cumsum(np.array(fp_flags, dtype=float))
    precision = tp / np.maximum(tp + fp_cum, 1e-12)
    recall = tp / max(float(total_gt), 1e-12)
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))


def evaluate_detections(
    pred_df: pd.DataFrame,
    gt_mapped_df: pd.DataFrame,
    iou_threshold: float = 0.5,
) -> dict[str, float]:
    pred_count = float(len(pred_df)) if pred_df is not None else 0.0
    if gt_mapped_df is None or gt_mapped_df.empty:
        return {
            "precision@0.5": float("nan"), "recall@0.5": float("nan"),
            "f1@0.5": float("nan"), "mean_iou": float("nan"),
            "ap@0.5": float("nan"), "duplicate_rate_eval": float("nan"),
            "tp": 0.0, "fp": 0.0, "fn": 0.0, "gt_count": 0.0,
            "pred_count_eval": pred_count,
        }

    gt_by_frame: dict[int, list] = {}
    for row in gt_mapped_df.itertuples(index=False):
        fidx = int(row.mapped_frame_index)
        gt_by_frame.setdefault(fidx, []).append((
            int(round(float(row.x_min))), int(round(float(row.y_min))),
            int(round(float(row.x_max))), int(round(float(row.y_max))),
        ))

    pred_items = []
    if pred_df is not None and not pred_df.empty:
        for row in pred_df.itertuples(index=False):
            pred_items.append({
                "frame_index": int(row.frame_index),
                "bbox": (int(row.x_min), int(row.y_min), int(row.x_max), int(row.y_max)),
                "score": float(row.candidate_score),
            })
    pred_items.sort(key=lambda x: x["score"], reverse=True)

    matched_flags = {fidx: [False] * len(boxes) for fidx, boxes in gt_by_frame.items()}
    tp_flags, fp_flags, matched_ious = [], [], []

    for pred in pred_items:
        fidx = pred["frame_index"]
        gt_boxes = gt_by_frame.get(fidx, [])
        used = matched_flags.get(fidx, [])
        best_iou, best_gt_idx = 0.0, -1
        for i, gt_box in enumerate(gt_boxes):
            if used[i]:
                continue
            iou = box_iou(pred["bbox"], gt_box)
            if iou > best_iou:
                best_iou, best_gt_idx = iou, i
        if best_gt_idx >= 0 and best_iou >= iou_threshold:
            used[best_gt_idx] = True
            tp_flags.append(1)
            fp_flags.append(0)
            matched_ious.append(best_iou)
        else:
            tp_flags.append(0)
            fp_flags.append(1)

    tp = int(sum(tp_flags))
    fp = int(sum(fp_flags))
    gt_count = int(len(gt_mapped_df))
    fn = int(max(0, gt_count - tp))
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, gt_count)
    f1 = 0.0 if (precision + recall) == 0 else (2 * precision * recall) / (precision + recall)
    mean_iou = float(np.mean(matched_ious)) if matched_ious else 0.0
    ap_05 = _compute_ap_from_flags(tp_flags, fp_flags, gt_count)

    dup_sum = 0
    if pred_df is not None and not pred_df.empty:
        for fidx, boxes in gt_by_frame.items():
            dup_sum += max(0, int((pred_df["frame_index"] == fidx).sum()) - len(boxes))
    duplicate_rate_eval = dup_sum / max(1, gt_count)

    return {
        "precision@0.5": float(precision), "recall@0.5": float(recall),
        "f1@0.5": float(f1), "mean_iou": float(mean_iou),
        "ap@0.5": float(ap_05), "duplicate_rate_eval": float(duplicate_rate_eval),
        "tp": float(tp), "fp": float(fp), "fn": float(fn),
        "gt_count": float(gt_count), "pred_count_eval": float(len(pred_items)),
    }


## 11. Inference pipeline and metrics summary


In [ ]:
def _is_border_cut(
    bbox: tuple[int, int, int, int], image_shape: tuple[int, ...], margin: int = 1
) -> bool:
    x1, y1, x2, y2 = bbox
    h, w = image_shape[:2]
    return x1 <= margin or y1 <= margin or x2 >= (w - 1 - margin) or y2 >= (h - 1 - margin)


def _draw_candidates(image: np.ndarray, candidates: list[Candidate]) -> np.ndarray:
    vis = image.copy()
    for cand in candidates:
        x1, y1, x2, y2 = cand.bbox
        color = (0, 255, 0) if cand.source == "yolo" else (0, 165, 255)
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
        cv2.putText(vis, f"{cand.source}|{cand.score:.2f}", (x1, max(0, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    return vis


def _apply_rotation(image: np.ndarray, rotation: str) -> np.ndarray:
    rot = (rotation or "none").lower()
    if rot == "cw":
        return cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    if rot == "ccw":
        return cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
    if rot == "180":
        return cv2.rotate(image, cv2.ROTATE_180)
    return image


def _resolve_rotation_mode(frame_records: list[dict[str, Any]], requested_mode: str) -> str:
    mode = (requested_mode or "none").strip().lower()
    return mode if mode in {"none", "cw", "ccw", "180"} else "ccw"


def _rotate_point(x: float, y: float, width: float, height: float, rotation: str) -> tuple[float, float]:
    rot = (rotation or "none").lower()
    if rot == "cw":
        return (height - 1.0 - y, x)
    if rot == "ccw":
        return (y, width - 1.0 - x)
    if rot == "180":
        return (width - 1.0 - x, height - 1.0 - y)
    return (x, y)


def _rotate_bbox(
    bbox: tuple[float, float, float, float], width: float, height: float, rotation: str
) -> tuple[float, float, float, float]:
    x1, y1, x2, y2 = bbox
    corners = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
    rc = [_rotate_point(x, y, width, height, rotation) for x, y in corners]
    return (min(p[0] for p in rc), min(p[1] for p in rc),
            max(p[0] for p in rc), max(p[1] for p in rc))


def _transform_gt_for_rotation(
    gt_df: pd.DataFrame, rotation: str, orig_width: int, orig_height: int
) -> pd.DataFrame:
    rot = (rotation or "none").lower()
    if gt_df is None or gt_df.empty or rot == "none":
        return gt_df
    out = gt_df.copy()
    new_boxes = [
        _rotate_bbox(
            (_to_float_series(out["x_min"])[i], _to_float_series(out["y_min"])[i],
             _to_float_series(out["x_max"])[i], _to_float_series(out["y_max"])[i]),
            float(orig_width), float(orig_height), rot,
        )
        for i in range(len(out))
    ]
    out["x_min"] = [b[0] for b in new_boxes]
    out["y_min"] = [b[1] for b in new_boxes]
    out["x_max"] = [b[2] for b in new_boxes]
    out["y_max"] = [b[3] for b in new_boxes]
    return out


def _bbox_center(box: tuple[int, int, int, int]) -> tuple[float, float]:
    x1, y1, x2, y2 = box
    return (0.5 * (x1 + x2), 0.5 * (y1 + y2))


def _bbox_diag(box: tuple[int, int, int, int]) -> float:
    x1, y1, x2, y2 = box
    return float(np.hypot(max(1, x2 - x1), max(1, y2 - y1)))


def _center_distance_norm(box_a: tuple[int, int, int, int], box_b: tuple[int, int, int, int]) -> float:
    ca = _bbox_center(box_a)
    cb = _bbox_center(box_b)
    dist = float(np.hypot(ca[0] - cb[0], ca[1] - cb[1]))
    scale = max(_bbox_diag(box_a), _bbox_diag(box_b), 1.0)
    return float(np.clip(dist / scale, 0.0, 1.0))


def _track_association_cost(
    box_a: tuple[int, int, int, int],
    box_b: tuple[int, int, int, int],
    alpha: float = TRACK_COST_ALPHA,
    beta: float = TRACK_COST_BETA,
) -> float:
    iou = box_iou(box_a, box_b)
    center_dist = _center_distance_norm(box_a, box_b)
    return float(alpha * (1.0 - iou) + beta * center_dist)


def _hungarian_assign(
    cost_matrix: np.ndarray,
    max_cost: float,
) -> list[tuple[int, int, float]]:
    """Return (row, col, cost) pairs with cost <= max_cost."""
    if cost_matrix.size == 0:
        return []
    from scipy.optimize import linear_sum_assignment

    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    pairs: list[tuple[int, int, float]] = []
    for r, c in zip(row_ind.tolist(), col_ind.tolist()):
        cst = float(cost_matrix[r, c])
        if cst <= float(max_cost):
            pairs.append((int(r), int(c), cst))
    return pairs


def _estimate_unique_price_tags(
    df: pd.DataFrame,
    max_gap_ms: int = TRACK_MAX_GAP_MS,
    cost_alpha: float = TRACK_COST_ALPHA,
    cost_beta: float = TRACK_COST_BETA,
    max_match_cost: float = TRACK_MAX_MATCH_COST,
) -> tuple[int, list[dict[str, Any]]]:
    """Greedy temporal tracks with per-frame Hungarian on active tracks."""
    if df is None or df.empty:
        return 0, []

    work = df.sort_values(["timestamp_ms", "candidate_score"], ascending=[True, False]).reset_index(drop=True)
    tracks: list[dict[str, Any]] = []
    next_track_id = 0

    for ts, group in work.groupby("timestamp_ms", sort=True):
        ts = int(ts)
        dets = [
            (int(r.x_min), int(r.y_min), int(r.x_max), int(r.y_max))
            for r in group.itertuples(index=False)
        ]
        if not dets:
            continue

        active_indices = [
            i for i, tr in enumerate(tracks)
            if ts - int(tr["last_ts"]) <= int(max_gap_ms)
        ]

        if not active_indices:
            for bbox in dets:
                tracks.append({
                    "track_id": next_track_id,
                    "bbox": bbox,
                    "last_ts": ts,
                    "count": 1,
                })
                next_track_id += 1
            continue

        n_det = len(dets)
        n_act = len(active_indices)
        cost = np.zeros((n_det, n_act), dtype=np.float64)
        for i, bbox in enumerate(dets):
            for j, tr_idx in enumerate(active_indices):
                cost[i, j] = _track_association_cost(
                    bbox,
                    tracks[tr_idx]["bbox"],
                    alpha=cost_alpha,
                    beta=cost_beta,
                )

        pairs = _hungarian_assign(cost, max_cost=max_match_cost)
        matched_det: set[int] = set()
        matched_act_local: set[int] = set()

        for det_i, act_j, _ in pairs:
            tr_idx = active_indices[act_j]
            tracks[tr_idx]["bbox"] = dets[det_i]
            tracks[tr_idx]["last_ts"] = ts
            tracks[tr_idx]["count"] = int(tracks[tr_idx]["count"]) + 1
            matched_det.add(det_i)
            matched_act_local.add(act_j)

        for det_i, bbox in enumerate(dets):
            if det_i in matched_det:
                continue
            tracks.append({
                "track_id": next_track_id,
                "bbox": bbox,
                "last_ts": ts,
                "count": 1,
            })
            next_track_id += 1

    return len(tracks), tracks


def assign_price_tag_track_ids(
    df: pd.DataFrame,
    max_gap_ms: int = TRACK_MAX_GAP_MS,
    cost_alpha: float = TRACK_COST_ALPHA,
    cost_beta: float = TRACK_COST_BETA,
    max_match_cost: float = TRACK_MAX_MATCH_COST,
) -> pd.DataFrame:
    """Add track_id per detection row using the same Hungarian association."""
    if df is None or df.empty:
        out = df.copy() if df is not None else pd.DataFrame()
        if not out.empty:
            out["track_id"] = pd.Series(dtype=int)
        return out

    out = df.copy()
    out["track_id"] = -1
    work = out.sort_values(["timestamp_ms", "candidate_score"], ascending=[True, False])
    tracks: list[dict[str, Any]] = []
    next_track_id = 0

    for ts, group in work.groupby("timestamp_ms", sort=True):
        ts = int(ts)
        row_ids = list(group.index)
        dets = [
            (int(r.x_min), int(r.y_min), int(r.x_max), int(r.y_max))
            for r in group.itertuples(index=False)
        ]

        active_indices = [
            i for i, tr in enumerate(tracks)
            if ts - int(tr["last_ts"]) <= int(max_gap_ms)
        ]

        if not active_indices:
            for orig_idx, bbox in zip(row_ids, dets):
                tracks.append({
                    "track_id": next_track_id,
                    "bbox": bbox,
                    "last_ts": ts,
                })
                out.at[orig_idx, "track_id"] = next_track_id
                next_track_id += 1
            continue

        n_det = len(dets)
        n_act = len(active_indices)
        cost = np.zeros((n_det, n_act), dtype=np.float64)
        for i, bbox in enumerate(dets):
            for j, tr_idx in enumerate(active_indices):
                cost[i, j] = _track_association_cost(
                    bbox,
                    tracks[tr_idx]["bbox"],
                    alpha=cost_alpha,
                    beta=cost_beta,
                )

        pairs = _hungarian_assign(cost, max_cost=max_match_cost)
        matched_det: set[int] = set()

        for det_i, act_j, _ in pairs:
            tr_idx = active_indices[act_j]
            tid = int(tracks[tr_idx]["track_id"])
            tracks[tr_idx]["bbox"] = dets[det_i]
            tracks[tr_idx]["last_ts"] = ts
            out.at[row_ids[det_i], "track_id"] = tid
            matched_det.add(det_i)

        for det_i, bbox in enumerate(dets):
            if det_i in matched_det:
                continue
            tracks.append({
                "track_id": next_track_id,
                "bbox": bbox,
                "last_ts": ts,
            })
            out.at[row_ids[det_i], "track_id"] = next_track_id
            next_track_id += 1

    return out


def _compute_proxy_and_business_metrics(
    result_df: pd.DataFrame,
    frame_stats: list[dict[str, Any]],
    processing_time_sec: float,
) -> dict[str, float]:
    frames_total = len(frame_stats)
    if frames_total == 0:
        return {
            "candidates_per_frame_mean": 0.0,
            "mean_quality_score": 0.0,
            "mean_sharpness": 0.0,
            "border_cut_rate": 0.0,
            "fallback_usage_rate": 0.0,
            "no_detection_frame_rate": 1.0,
            "processing_time_sec": float(processing_time_sec),
            "effective_fps": 0.0,
            "unique_price_tags_est": 0.0,
            "duplicate_rate_video": 0.0,
            "actionable_frame_rate": 0.0,
            "frame_sharpness_score_mean": 0.0,
            "frame_custom_score_mean": 0.0,
            "ocr_filtered_ratio": 0.0,
            "window_selected_ratio": 0.0,
        }
    total_cands = sum(int(s["detections"]) for s in frame_stats)
    fallback_rate = float(np.mean([1.0 if s["fallback_used"] else 0.0 for s in frame_stats]))
    no_det_rate = float(np.mean([1.0 if s["detections"] == 0 else 0.0 for s in frame_stats]))
    filtered_rate = float(np.mean([1.0 if s.get("filtered_out", False) else 0.0 for s in frame_stats]))
    selected_rate = float(np.mean([1.0 if s.get("selected_for_ocr", True) else 0.0 for s in frame_stats]))
    unique_est, _ = _estimate_unique_price_tags(result_df)
    dup_rate = max(0.0, (float(total_cands) - float(unique_est)) / max(1.0, float(unique_est)))
    return {
        "candidates_per_frame_mean": float(total_cands / max(1, frames_total)),
        "mean_quality_score": float(result_df["quality_score"].mean()) if not result_df.empty else 0.0,
        "mean_sharpness": float(result_df["sharpness"].mean()) if not result_df.empty else 0.0,
        "border_cut_rate": float(result_df["border_cut"].mean()) if (not result_df.empty and "border_cut" in result_df.columns) else 0.0,
        "fallback_usage_rate": fallback_rate,
        "no_detection_frame_rate": no_det_rate,
        "processing_time_sec": float(processing_time_sec),
        "effective_fps": float(frames_total / max(processing_time_sec, 1e-9)),
        "unique_price_tags_est": float(unique_est),
        "duplicate_rate_video": dup_rate,
        "actionable_frame_rate": float(1.0 - no_det_rate),
        "frame_sharpness_score_mean": float(np.mean([s["frame_sharpness_score"] for s in frame_stats])),
        "frame_custom_score_mean": float(np.mean([s["frame_custom_score"] for s in frame_stats])),
        "ocr_filtered_ratio": filtered_rate,
        "window_selected_ratio": selected_rate,
    }


# def _compute_tenengrad(image_bgr: np.ndarray) -> float:
#     gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
#     gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
#     gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
#     grad_mag_sq = gx * gx + gy * gy
#     return float(np.mean(grad_mag_sq))


def _frame_topk_sort_key(rec: dict[str, Any]) -> tuple[float, float]:
    prep_scores = rec.get("prep_scores") or {}
    return (
        float(prep_scores.get("custom_frame_score", 0.0)),
        float(rec.get("tenengrad", 0.0)),
    )


def _pick_diverse_topk_in_ranked_list(
    ranked: list[dict[str, Any]],
    top_k: int,
    similarity_thr: float,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Greedy diverse top-K: high score first, skip near-duplicates (cosine >= thr)."""
    selected: list[dict[str, Any]] = []
    audit: list[dict[str, Any]] = []
    top_k = max(1, int(top_k))

    for rank, rec in enumerate(ranked, start=1):
        sig = rec.get("sim_signature")
        if sig is None:
            image_rot = rec.get("image_rot")
            sig = _frame_similarity_signature(image_rot) if image_rot is not None else None
            rec["sim_signature"] = sig

        max_sim = 0.0
        if selected and sig is not None:
            for kept in selected:
                kept_sig = kept.get("sim_signature")
                if kept_sig is None:
                    continue
                max_sim = max(max_sim, float(np.dot(sig, kept_sig)))

        if len(selected) >= top_k:
            pick = False
            reason = "topk_quota_full"
        elif max_sim >= float(similarity_thr):
            pick = False
            reason = "too_similar"
        else:
            pick = True
            reason = "selected"
            selected.append(rec)

        audit.append({
            "rec": rec,
            "rank_in_window": int(rank),
            "selected_for_yolo": int(pick),
            "rejection_reason": reason,
            "max_similarity_to_selected": float(max_sim),
        })

    return selected, audit


def _select_frames_by_temporal_windows(
    frame_candidates: list[dict[str, Any]],
    window_sec: float,
    top_k_per_window: int,
    similarity_thr: float = FRAME_SIMILARITY_THR,
) -> list[dict[str, Any]]:
    if not frame_candidates:
        return []
    window_ms = max(1, int(round(window_sec * 1000.0)))
    top_k = max(1, int(top_k_per_window))

    buckets: dict[int, list[dict[str, Any]]] = {}
    for rec in frame_candidates:
        ts = int(rec["timestamp_ms"])
        win_id = ts // window_ms
        buckets.setdefault(win_id, []).append(rec)

    selected: list[dict[str, Any]] = []
    for _, items in sorted(buckets.items(), key=lambda kv: kv[0]):
        ranked = sorted(items, key=_frame_topk_sort_key, reverse=True)
        window_selected, _ = _pick_diverse_topk_in_ranked_list(
            ranked=ranked,
            top_k=top_k,
            similarity_thr=similarity_thr,
        )
        selected.extend(window_selected)

    selected.sort(key=lambda r: int(r["frame_index"]))
    return selected


def build_temporal_topk_window_report(
    frame_candidates: list[dict[str, Any]],
    window_sec: float,
    top_k_per_window: int,
    similarity_thr: float = FRAME_SIMILARITY_THR,
) -> pd.DataFrame:
    """Per window: frames sorted by score + diverse top-K selection audit."""
    if not frame_candidates:
        return pd.DataFrame()

    window_ms = max(1, int(round(window_sec * 1000.0)))
    top_k = max(1, int(top_k_per_window))

    buckets: dict[int, list[dict[str, Any]]] = {}
    for rec in frame_candidates:
        ts = int(rec["timestamp_ms"])
        win_id = int(ts // window_ms)
        buckets.setdefault(win_id, []).append(rec)

    rows: list[dict[str, Any]] = []
    for win_id, items in sorted(buckets.items()):
        ranked = sorted(items, key=_frame_topk_sort_key, reverse=True)
        window_start_ms = int(win_id * window_ms)
        window_end_ms = int(window_start_ms + window_ms)
        _, audit = _pick_diverse_topk_in_ranked_list(
            ranked=ranked,
            top_k=top_k,
            similarity_thr=similarity_thr,
        )

        for entry in audit:
            rec = entry["rec"]
            prep_scores = rec.get("prep_scores") or {}
            frame_idx = int(rec["frame_index"])
            timestamp_ms = int(rec["timestamp_ms"])
            rows.append({
                "window_id": int(win_id),
                "window_start_ms": window_start_ms,
                "window_end_ms": window_end_ms,
                "rank_in_window": int(entry["rank_in_window"]),
                "selected_for_yolo": int(entry["selected_for_yolo"]),
                "rejection_reason": str(entry["rejection_reason"]),
                "max_similarity_to_selected": float(entry["max_similarity_to_selected"]),
                "frame_index": frame_idx,
                "timestamp_ms": timestamp_ms,
                "timestamp_sec": round(timestamp_ms / 1000.0, 3),
                "custom_frame_score": float(prep_scores.get("custom_frame_score", 0.0)),
                "sharpness_score": float(prep_scores.get("sharpness_score", 0.0)),
                "tenengrad": float(rec.get("tenengrad", 0.0)),
                "glare_ratio": float(prep_scores.get("glare_ratio", 0.0)),
                "frame_path": str(rec.get("frame_path", "")),
            })

    report_df = pd.DataFrame(rows)
    if not report_df.empty:
        report_df = report_df.sort_values(
            ["window_id", "rank_in_window", "frame_index"],
            ascending=[True, True, True],
        ).reset_index(drop=True)
    return report_df


def show_temporal_topk_window_report(
    report_df: pd.DataFrame,
    window_sec: float,
    top_k_per_window: int,
    max_windows: int | None = None,
) -> None:
    """Print sorted frames per temporal window and visualize selected vs rejected."""
    if report_df is None or report_df.empty:
        print("[TOP-K VIZ] Empty temporal top-K report.")
        return

    window_ids = sorted(report_df["window_id"].unique().tolist())
    if max_windows is not None:
        window_ids = window_ids[: int(max_windows)]

    print(
        f"\n[TOP-K VIZ] Temporal windows: {len(window_ids)} shown "
        f"(window={window_sec}s, top_k={top_k_per_window})"
    )

    for win_id in window_ids:
        part = report_df[report_df["window_id"] == win_id].copy()
        if part.empty:
            continue

        w_start = int(part.iloc[0]["window_start_ms"])
        w_end = int(part.iloc[0]["window_end_ms"])
        n_sel = int(part["selected_for_yolo"].sum())
        print(
            f"\n--- Window {win_id}: {w_start}–{w_end} ms | "
            f"frames={len(part)} | selected={n_sel}/{top_k_per_window} ---"
        )
        table_cols = [
            "rank_in_window",
            "selected_for_yolo",
            "rejection_reason",
            "max_similarity_to_selected",
            "frame_index",
            "timestamp_sec",
            "custom_frame_score",
            "sharpness_score",
            "glare_ratio",
        ]
        try:
            from IPython.display import display as ipy_display

            ipy_display(part[table_cols])
        except Exception:
            print(part[table_cols].to_string(index=False))

        rows_for_plot = list(part.itertuples(index=False))
        if not rows_for_plot:
            continue

        cols_n = min(3, len(rows_for_plot))
        rows_n = int(math.ceil(len(rows_for_plot) / cols_n))
        fig, axes = plt.subplots(rows_n, cols_n, figsize=(4.5 * cols_n, 3.8 * rows_n))
        if rows_n == 1 and cols_n == 1:
            axes_flat = [axes]
        else:
            axes_flat = np.array(axes).reshape(-1)

        for ax in axes_flat[len(rows_for_plot):]:
            ax.axis("off")

        for ax, row in zip(axes_flat, rows_for_plot):
            img = cv2.imread(str(row.frame_path))
            if img is None:
                ax.set_title(f"missing: {row.frame_path}")
                ax.axis("off")
                continue
            tag = "SELECTED" if int(row.selected_for_yolo) == 1 else str(row.rejection_reason)
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.set_title(
                f"#{int(row.rank_in_window)} {tag}\n"
                f"score={float(row.custom_frame_score):.3f} | "
                f"sim={float(row.max_similarity_to_selected):.2f} | t={float(row.timestamp_sec):.2f}s",
                fontsize=9,
            )
            ax.axis("off")

        fig.suptitle(
            f"Window {win_id}: {w_start/1000:.1f}–{w_end/1000:.1f}s "
            f"(sorted by custom_frame_score, top-{top_k_per_window} selected)",
            fontsize=12,
        )
        plt.tight_layout()
        plt.show()


def process_video_baseline(
    video_path: str,
    output_dir: str,
    sample_fps: float = 2.0,
    min_sample_fps: float = 1.0,
    flow_magnitude_thr: float = 0.15,
    flow_resize_width: int = 320,
    temporal_window_sec: float = 3.0,
    top_k_frames_per_window: int = 3,
    # max_glare_ratio_for_ocr: float = 0.08,
    # tenengrad_median_factor: float = 0.5,
    # min_brightness_for_ocr: float = 40.0,
    # max_brightness_for_ocr: float = 220.0,
    max_frames: int | None = None,
    gt_df: pd.DataFrame | None = None,
    frame_rotation: str = "none",
) -> tuple[pd.DataFrame, dict[str, Any]]:
    t0 = time.perf_counter()

    out_dir = Path(output_dir)
    frames_dir = out_dir / "frames"
    crops_dir = out_dir / "crops"
    debug_dir = out_dir / "debug_frames"
    preproc_dir = out_dir / "preprocessed_frames"
    for d in [out_dir, frames_dir, crops_dir, debug_dir, preproc_dir]:
        d.mkdir(parents=True, exist_ok=True)

    video_info = inspect_video(video_path)
    print("Video info:")
    for k, v in video_info.items():
        print(f"  {k}: {v}")

    frame_records = extract_frames(
        video_path=video_path,
        frames_dir=frames_dir,
        sample_fps=sample_fps,
        min_sample_fps=min_sample_fps,
        flow_magnitude_thr=flow_magnitude_thr,
        flow_resize_width=flow_resize_width,
        max_frames=max_frames,
    )
    print(f"Extracted frames: {len(frame_records)}")

    resolved_rotation = _resolve_rotation_mode(frame_records, frame_rotation)
    print(f"Frame rotation: requested={frame_rotation}, resolved={resolved_rotation}")

    gt_for_eval = gt_df
    if gt_df is not None and not gt_df.empty and resolved_rotation != "none":
        gt_for_eval = _transform_gt_for_rotation(
            gt_df=gt_df, rotation=resolved_rotation,
            orig_width=int(video_info.get("width", 0) or 0),
            orig_height=int(video_info.get("height", 0) or 0),
        )

    gt_mapped_df: pd.DataFrame | None = None
    if gt_for_eval is not None and not gt_for_eval.empty:
        gt_mapped_df, _ = _map_gt_to_extracted_frames(gt_for_eval, frame_records, sample_fps)

    debug_records: list[dict[str, Any]] = []
    frame_stats: list[dict[str, Any]] = []
    frame_quality_records: list[dict[str, Any]] = []
    video_filename = Path(video_path).name

    frame_candidates: list[dict[str, Any]] = []
    for rec in tqdm(frame_records, desc="Frame quality prepass"):
        frame_idx = int(rec["frame_index"])
        timestamp_ms = int(rec["timestamp_ms"])

        image = cv2.imread(rec["frame_path"])
        if image is None:
            continue

        image_rot = _apply_rotation(image, resolved_rotation)
        prep = preprocess_frame_bundle(image_rot)
        raw_metrics_rot = compute_frame_raw_metrics(image_rot)
        raw_metrics_prep = compute_frame_raw_metrics(prep["ocr_ready_bgr"])

        frame_candidates.append({
            "frame_index": frame_idx,
            "timestamp_ms": timestamp_ms,
            "frame_path": rec["frame_path"],
            "image_rot": image_rot,
            "prep": prep,
            "raw_metrics_rot": raw_metrics_rot,
            "raw_metrics_prep": raw_metrics_prep,
        })

    prep_percentile_stats = build_video_percentile_stats(
        [r["raw_metrics_prep"] for r in frame_candidates],
        p_low=SCORE_PERCENTILE_LOW,
        p_high=SCORE_PERCENTILE_HIGH,
    )
    rot_percentile_stats = build_video_percentile_stats(
        [r["raw_metrics_rot"] for r in frame_candidates],
        p_low=SCORE_PERCENTILE_LOW,
        p_high=SCORE_PERCENTILE_HIGH,
    )

    for rec in frame_candidates:
        rec["raw_scores"] = raw_metrics_to_frame_scores(rec["raw_metrics_rot"], rot_percentile_stats)
        rec["prep_scores"] = raw_metrics_to_frame_scores(rec["raw_metrics_prep"], prep_percentile_stats)
        rec["tenengrad"] = float(rec["raw_metrics_prep"]["sharpness"])
        rec["custom_frame_score"] = float(rec["prep_scores"]["custom_frame_score"])
        rec["sim_signature"] = _frame_similarity_signature(
            rec["image_rot"], size=FRAME_SIMILARITY_SIZE
        )

    # HARD FRAME FILTERS (disabled) — keep metrics in prepass for debug only
    # tenengrad_median = float(np.median([r["tenengrad"] for r in frame_candidates])) if frame_candidates else 0.0
    # tenengrad_thr = tenengrad_median * max(float(tenengrad_median_factor), 0.0)
    #
    # prefiltered_candidates: list[dict[str, Any]] = []
    # for rec in frame_candidates:
    #     prep_scores = rec["prep_scores"]
    #     glare_ratio = float(prep_scores.get("glare_ratio", 0.0))
    #     brightness = float(prep_scores.get("brightness", 0.0))
    #     tenengrad = float(rec.get("tenengrad", 0.0))
    #
    #     fail_glare = glare_ratio > float(max_glare_ratio_for_ocr)
    #     fail_blur = tenengrad < float(tenengrad_thr)
    #     fail_brightness = brightness < float(min_brightness_for_ocr) or brightness > float(max_brightness_for_ocr)
    #     filtered_out = bool(fail_glare or fail_blur or fail_brightness)
    #
    #     rec["filtered_out"] = filtered_out
    #     rec["filter_reason"] = "|".join([
    #         "glare" if fail_glare else "",
    #         "blur" if fail_blur else "",
    #         "brightness" if fail_brightness else "",
    #     ]).strip("|")
    #
    #     if not filtered_out:
    #         prefiltered_candidates.append(rec)

    for rec in frame_candidates:
        rec["filtered_out"] = False
        rec["filter_reason"] = ""

    top_k_pool = [rec for rec in frame_candidates if not rec.get("filtered_out", False)]
    selected_records = _select_frames_by_temporal_windows(
        frame_candidates=top_k_pool,
        window_sec=temporal_window_sec,
        top_k_per_window=top_k_frames_per_window,
        similarity_thr=FRAME_SIMILARITY_THR,
    )
    selected_ids = {
        (int(r["frame_index"]), int(r["timestamp_ms"]))
        for r in selected_records
    }
    print(
        f"Temporal top-K: {len(selected_ids)}/{len(frame_candidates)} frames "
        f"selected for YOLO (window={temporal_window_sec}s, k={top_k_frames_per_window}, "
        f"sim_thr={FRAME_SIMILARITY_THR})"
    )

    topk_report_df = build_temporal_topk_window_report(
        frame_candidates=top_k_pool,
        window_sec=temporal_window_sec,
        top_k_per_window=top_k_frames_per_window,
        similarity_thr=FRAME_SIMILARITY_THR,
    )
    topk_report_csv_path = out_dir / "temporal_topk_windows.csv"
    topk_report_df.to_csv(topk_report_csv_path, index=False)
    print(f"Temporal top-K report saved: {topk_report_csv_path}")

    if SHOW_TEMPORAL_TOPK_DEBUG:
        show_temporal_topk_window_report(
            report_df=topk_report_df,
            window_sec=temporal_window_sec,
            top_k_per_window=top_k_frames_per_window,
            max_windows=TEMPORAL_TOPK_DEBUG_MAX_WINDOWS,
        )

    for rec in frame_candidates:
        frame_idx = int(rec["frame_index"])
        timestamp_ms = int(rec["timestamp_ms"])
        selected_for_ocr = (frame_idx, timestamp_ms) in selected_ids

        image_rot = rec["image_rot"]
        prep = rec["prep"]
        raw_scores = rec["raw_scores"]
        prep_scores = rec["prep_scores"]
        tenengrad = float(rec["tenengrad"])

        rotated_path = preproc_dir / f"rotated_{frame_idx:08d}_{timestamp_ms:010d}ms.jpg"
        preproc_path = preproc_dir / f"preproc_{frame_idx:08d}_{timestamp_ms:010d}ms.jpg"
        if SAVE_PREPROCESSED_FRAMES:
            cv2.imwrite(str(rotated_path), image_rot)
            cv2.imwrite(str(preproc_path), prep["ocr_ready_bgr"])

        frame_quality_records.append({
            "video_filename": video_filename,
            "frame_index": frame_idx,
            "timestamp_ms": timestamp_ms,
            "frame_path": rec["frame_path"],
            "rotated_frame_path": str(rotated_path),
            "preprocessed_frame_path": str(preproc_path),
            "raw_sharpness": raw_scores["sharpness"],
            "raw_brightness": raw_scores["brightness"],
            "raw_contrast": raw_scores["contrast"],
            "raw_glare_ratio": raw_scores.get("glare_ratio", 0.0),
            "prep_sharpness": prep_scores["sharpness"],
            "prep_brightness": prep_scores["brightness"],
            "prep_contrast": prep_scores["contrast"],
            "prep_glare_ratio": prep_scores.get("glare_ratio", 0.0),
            "tenengrad": tenengrad,
            # "tenengrad_median": tenengrad_median,
            # "tenengrad_threshold": tenengrad_thr,
            "filtered_out": int(rec.get("filtered_out", False)),
            "filter_reason": rec.get("filter_reason", ""),
            "selected_for_ocr": int(selected_for_ocr),
            "frame_sharpness_score": prep_scores["sharpness_score"],
            "frame_custom_score": prep_scores["custom_frame_score"],
        })

        fallback_used = False
        candidates: list[Candidate] = []
        if selected_for_ocr:
            # YOLO detection on preprocessed image; fallback to contours if nothing found
            candidates = detect_with_yolo(prep["ocr_ready_bgr"])
            if not candidates:
                candidates = find_rectangular_candidates(
                    image_rot,
                    min_area_ratio=YOLO_MIN_AREA_RATIO,
                    max_area_ratio=YOLO_MAX_AREA_RATIO,
                    aspect_ratio_range=(YOLO_ASPECT_RATIO_MIN, YOLO_ASPECT_RATIO_MAX),
                    nms_iou=YOLO_IOU_NMS,
                )
                fallback_used = len(candidates) > 0

            debug_img = _draw_candidates(image_rot, candidates)
            debug_frame_path = debug_dir / f"debug_{frame_idx:08d}_{timestamp_ms:010d}ms.jpg"
            cv2.imwrite(str(debug_frame_path), debug_img)

            for cand_id, cand in enumerate(candidates):
                x1, y1, x2, y2 = cand.bbox
                crop = image_rot[y1:y2, x1:x2]
                if crop is None or crop.size == 0:
                    continue
                crop_path = crops_dir / f"crop_{frame_idx:08d}_{timestamp_ms:010d}ms_{cand_id:03d}.jpg"
                cv2.imwrite(str(crop_path), crop)
                quality = compute_crop_quality(crop, candidate_score=float(cand.score))
                border_cut = _is_border_cut(cand.bbox, image_rot.shape)
                debug_records.append({
                    "video_filename": video_filename,
                    "frame_index": frame_idx,
                    "timestamp_ms": timestamp_ms,
                    "candidate_id": cand_id,
                    "crop_path": str(crop_path),
                    "debug_frame_path": str(debug_frame_path),
                    "rotated_frame_path": str(rotated_path),
                    "preprocessed_frame_path": str(preproc_path),
                    "x_min": int(x1),
                    "y_min": int(y1),
                    "x_max": int(x2),
                    "y_max": int(y2),
                    "candidate_score": float(cand.score),
                    "quality_score": float(quality["quality_score"]),
                    "sharpness": float(quality["sharpness"]),
                    "brightness": float(quality["brightness"]),
                    "contrast": float(quality["contrast"]),
                    "matched_texts": "",
                    "source": cand.source,
                    "border_cut": int(border_cut),
                    "frame_sharpness_score": prep_scores["sharpness_score"],
                    "frame_custom_score": prep_scores["custom_frame_score"],
                })

        frame_stats.append({
            "frame_index": frame_idx,
            "timestamp_ms": timestamp_ms,
            "detections": len(candidates),
            "fallback_used": bool(fallback_used),
            "filtered_out": bool(rec.get("filtered_out", False)),
            "selected_for_ocr": bool(selected_for_ocr),
            "frame_sharpness_score": prep_scores["sharpness_score"],
            "frame_custom_score": prep_scores["custom_frame_score"],
        })

    result_df = pd.DataFrame(debug_records)
    required_cols = [
        "video_filename", "frame_index", "timestamp_ms", "candidate_id",
        "crop_path", "debug_frame_path", "x_min", "y_min", "x_max", "y_max",
        "candidate_score", "quality_score", "sharpness", "brightness", "contrast",
        "matched_texts", "source", "frame_sharpness_score", "frame_custom_score",
    ]
    for col in required_cols:
        if col not in result_df.columns:
            result_df[col] = []

    if not result_df.empty:
        for score_col, rank_col in [("quality_score", "rank_by_quality"),
                                     ("sharpness", "rank_by_sharpness"),
                                     ("candidate_score", "rank_by_candidate")]:
            result_df[rank_col] = result_df.groupby("frame_index")[score_col].rank(ascending=False, method="first")
        result_df["is_best_by_quality"] = (result_df["rank_by_quality"] == 1).astype(int)
        result_df["is_best_by_sharpness"] = (result_df["rank_by_sharpness"] == 1).astype(int)
        result_df["is_best_by_candidate"] = (result_df["rank_by_candidate"] == 1).astype(int)

    debug_csv_path = out_dir / "debug_candidates.csv"
    result_df.to_csv(debug_csv_path, index=False)
    frame_quality_df = pd.DataFrame(frame_quality_records)
    frame_quality_csv_path = out_dir / "frame_quality_debug.csv"
    frame_quality_df.to_csv(frame_quality_csv_path, index=False)

    elapsed = time.perf_counter() - t0
    proxy_metrics = _compute_proxy_and_business_metrics(result_df, frame_stats, elapsed)

    if gt_mapped_df is not None and not gt_mapped_df.empty:
        eval_pred_df = (
            result_df[["frame_index", "candidate_score", "x_min", "y_min", "x_max", "y_max"]].copy()
            if not result_df.empty
            else pd.DataFrame(columns=["frame_index", "candidate_score", "x_min", "y_min", "x_max", "y_max"])
        )
        full_metrics = evaluate_detections(eval_pred_df, gt_mapped_df, iou_threshold=0.5)
    else:
        full_metrics = {k: float("nan") for k in [
            "precision@0.5", "recall@0.5", "f1@0.5", "mean_iou", "ap@0.5",
            "duplicate_rate_eval", "tp", "fp", "fn", "gt_count", "pred_count_eval",
        ]}

    metrics_summary: dict[str, Any] = {
        "video_path": video_path,
        "output_dir": str(out_dir),
        "frame_rotation_requested": frame_rotation,
        "frame_rotation_resolved": resolved_rotation,
        "sampling_mode": "adaptive_optical_flow",
        "sampling_params": {
            "max_sample_fps": float(sample_fps),
            "min_sample_fps": float(min_sample_fps),
            "flow_magnitude_thr": float(flow_magnitude_thr),
            "flow_resize_width": int(flow_resize_width),
        },
        "frame_scoring": {
            "mode": "percentile_roi_tenengrad",
            "roi_y_start_frac": float(SCORE_ROI_Y_START_FRAC),
            "roi_x_start_frac": float(SCORE_ROI_X_START_FRAC),
            "roi_x_end_frac": float(SCORE_ROI_X_END_FRAC),
            "percentile_low": float(SCORE_PERCENTILE_LOW),
            "percentile_high": float(SCORE_PERCENTILE_HIGH),
            "prep_percentiles": {
                k: {"p10": float(v[0]), "p90": float(v[1])} for k, v in prep_percentile_stats.items()
            },
        },
        "frame_filtering": {
            "temporal_window_sec": float(temporal_window_sec),
            "top_k_frames_per_window": int(top_k_frames_per_window),
            "temporal_top_k_enabled": True,
            "frames_selected_for_yolo": len(selected_ids),
            "temporal_topk_report_csv": str(topk_report_csv_path),
            "frame_similarity_thr": float(FRAME_SIMILARITY_THR),
            "frame_similarity_size": int(FRAME_SIMILARITY_SIZE),
            "hard_frame_filters_enabled": False,
            # "max_glare_ratio_for_ocr": float(max_glare_ratio_for_ocr),
            # "tenengrad_median_factor": float(tenengrad_median_factor),
            # "tenengrad_median": float(tenengrad_median),
            # "tenengrad_threshold": float(tenengrad_thr),
            # "min_brightness_for_ocr": float(min_brightness_for_ocr),
            # "max_brightness_for_ocr": float(max_brightness_for_ocr),
        },
        "yolo_weights": YOLO_FINETUNED_MODEL or YOLO_BASE_MODEL,
        "frames_processed": len(frame_stats),
        "detections_total": int(len(result_df)),
        "full_detection_metrics": full_metrics,
        "proxy_business_metrics": proxy_metrics,
    }

    flat_metrics = {
        "frames_processed": float(metrics_summary["frames_processed"]),
        "detections_total": float(metrics_summary["detections_total"]),
    }
    for k, v in {**full_metrics, **proxy_metrics}.items():
        flat_metrics[k] = float(v) if pd.notna(v) else np.nan

    metrics_df = pd.DataFrame([{"metric": k, "value": v} for k, v in flat_metrics.items()])
    metrics_csv_path = out_dir / "metrics_summary.csv"
    metrics_json_path = out_dir / "metrics_summary.json"
    metrics_df.to_csv(metrics_csv_path, index=False)
    with metrics_json_path.open("w", encoding="utf-8") as f:
        json.dump(metrics_summary, f, ensure_ascii=False, indent=2)

    return result_df, {
        "metrics_summary": metrics_summary, "metrics_df": metrics_df,
        "debug_csv_path": str(debug_csv_path),
        "metrics_csv_path": str(metrics_csv_path),
        "metrics_json_path": str(metrics_json_path),
        "frame_quality_csv_path": str(frame_quality_csv_path),
        "output_dir": str(out_dir),
    }


## 12. Run baseline and Metrics Summary


In [ ]:
def discover_video_jobs(
    data_root: str,
    case_filter: str | None = None,
    include_unlabeled: bool = True,
) -> list[dict[str, str | None]]:
    root = Path(data_root)
    if not root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {data_root}")
    jobs: list[dict[str, str | None]] = []
    cf = case_filter.lower().strip() if isinstance(case_filter, str) and case_filter.strip() else None
    for mp4_path in sorted(root.rglob("*.mp4")):
        if any(p.startswith("baseline_yolo_candidates") or p == "yolo_runs" for p in mp4_path.parts):
            continue
        if not include_unlabeled and any(p.lower() == "unlabeled" for p in mp4_path.parts):
            continue
        if cf and cf not in str(mp4_path).lower():
            continue
        gt_candidate = mp4_path.with_suffix(".csv")
        gt_path = str(gt_candidate) if gt_candidate.exists() else None
        case_id = f"{mp4_path.parent.name}__{mp4_path.stem}" if mp4_path.parent.name else mp4_path.stem
        jobs.append({"case_id": case_id, "video_path": str(mp4_path), "gt_path": gt_path})
    return jobs


def _safe_case_name(text: str) -> str:
    name = re.sub(r"[^a-zA-Z0-9_.-]+", "_", text)
    return name.strip("_") or "case"


def _split_finetune_test(
    labeled_jobs: list[dict],
    holdout_ratio: float,
    test_video_names: list[str] | None,
) -> tuple[list[dict], list[dict]]:
    if not labeled_jobs:
        return [], []

    if test_video_names is not None:
        test_stems = {n.lower() for n in test_video_names}
        finetune = [j for j in labeled_jobs if Path(j["video_path"]).stem.lower() not in test_stems]
        test = [j for j in labeled_jobs if Path(j["video_path"]).stem.lower() in test_stems]
        return finetune, test

    # Auto split: last holdout_ratio videos → test (sorted by name for reproducibility)
    sorted_jobs = sorted(labeled_jobs, key=lambda j: Path(j["video_path"]).stem)
    n_test = max(1, int(len(sorted_jobs) * holdout_ratio))
    n_finetune = len(sorted_jobs) - n_test
    if n_finetune < 1:
        # Not enough videos to split — warn and return all as both
        print("[WARN] Only 1 labeled video — cannot split by video. "
              "Fine-tuning and test will overlap. Metrics will be on-train.")
        return sorted_jobs, sorted_jobs
    return sorted_jobs[:n_finetune], sorted_jobs[n_finetune:]


# ── Discover jobs ─────────────────────────────────────────────────────────────
run_mode = str(RUN_MODE).strip().lower()
if run_mode not in {"single", "batch"}:
    raise ValueError(f"RUN_MODE must be 'single' or 'batch', got: {RUN_MODE}")

if run_mode == "single":
    if not INPUT_VIDEO_PATH:
        raise ValueError("INPUT_VIDEO_PATH is empty in single mode")
    jobs = [{"case_id": _safe_case_name(Path(INPUT_VIDEO_PATH).stem),
             "video_path": INPUT_VIDEO_PATH, "gt_path": GT_ANNOTATIONS_PATH}]
else:
    jobs = discover_video_jobs(data_root=DATA_ROOT, case_filter=CASE_FILTER, include_unlabeled=INCLUDE_UNLABELED)

if not jobs:
    raise RuntimeError("No video jobs found. Check DATA_ROOT / CASE_FILTER / RUN_MODE.")

print(f"RUN_MODE={run_mode} | jobs found: {len(jobs)}")

# ── Train / test split ────────────────────────────────────────────────────────
labeled_jobs = [j for j in jobs if j.get("gt_path")]
unlabeled_jobs = [j for j in jobs if not j.get("gt_path")]

finetune_jobs, test_jobs = _split_finetune_test(
    labeled_jobs, FINETUNE_HOLDOUT_RATIO, TEST_VIDEO_NAMES
)

finetune_stems = {Path(j["video_path"]).stem for j in finetune_jobs}
test_stems = {Path(j["video_path"]).stem for j in test_jobs}

print(f"\nLabeled videos: {len(labeled_jobs)} total")
print(f"  finetune ({len(finetune_jobs)}): {sorted(finetune_stems)}")
print(f"  test     ({len(test_jobs)}): {sorted(test_stems)}")
print(f"  unlabeled: {len(unlabeled_jobs)}")

# ── Fine-tune (optional) ──────────────────────────────────────────────────────
if ENABLE_FINETUNE:
    if finetune_jobs:
        print(f"\nBuilding YOLO dataset from {len(finetune_jobs)} finetune video(s)...")
        data_yaml = build_yolo_dataset(
            gt_csv_paths=[j["gt_path"] for j in finetune_jobs],
            video_paths=[j["video_path"] for j in finetune_jobs],
            out_dir=FINETUNE_DATA_DIR,
            rotation=FRAME_ROTATION_MODE,
            split_ratio=FINETUNE_SPLIT_RATIO,
        )
        print(f"\nFine-tuning {YOLO_BASE_MODEL} on: {data_yaml}")
        YOLO_FINETUNED_MODEL = run_finetune(
            data_yaml=data_yaml,
            base_model=YOLO_BASE_MODEL,
            epochs=FINETUNE_EPOCHS,
            batch=FINETUNE_BATCH,
            img_size=FINETUNE_IMG_SIZE,
            patience=FINETUNE_PATIENCE,
            run_dir=FINETUNE_RUN_DIR,
            train_device=TRAIN_DEVICE,
        )
        _yolo_model = None  # reset so next call loads fine-tuned weights
        show_train_batches(FINETUNE_RUN_DIR, name="price_tag_ft")
    else:
        print("[WARN] ENABLE_FINETUNE=True but no finetune videos available.")
else:
    print("Fine-tuning disabled. Using base/pre-specified YOLO model.")


In [ ]:
# ── Inference ─────────────────────────────────────────────────────────────────
# Run on: test videos (with GT) + unlabeled videos
# Finetune-only videos are skipped from inference to keep metrics honest.
# If finetune==test (1-video fallback), all jobs run but metrics are marked on-train.
_test_for_infer = test_jobs[:1]          # максимум 1 размеченное
_unlabeled_for_infer = unlabeled_jobs[:1]  # максимум 1 неразмеченное
inference_jobs = _test_for_infer + _unlabeled_for_infer

# Deduplicate (finetune==test fallback may cause duplicates)
seen = set()
inference_jobs_dedup = []
for j in inference_jobs:
    key = j["video_path"]
    if key not in seen:
        seen.add(key)
        inference_jobs_dedup.append(j)

print(f"\nInference will run on {len(inference_jobs_dedup)} video(s):")
for j in inference_jobs_dedup:
    role = "test" if Path(j["video_path"]).stem in test_stems else "unlabeled"
    print(f"  [{role}] {j['case_id']}")

all_run_infos: list[dict[str, Any]] = []
all_result_parts: list[pd.DataFrame] = []
all_frame_quality_parts: list[pd.DataFrame] = []
summary_rows: list[dict[str, Any]] = []

for idx, job in enumerate(inference_jobs_dedup, start=1):
    case_id = str(job["case_id"])
    video_path = str(job["video_path"])
    gt_path = job.get("gt_path")
    is_test = Path(video_path).stem in test_stems

    print("\n" + "=" * 80)
    print(f"[{idx}/{len(inference_jobs_dedup)}] Processing case: {case_id}  [{'TEST' if is_test else 'UNLABELED'}]")

    gt_annotations_df = None
    if gt_path and is_test:
        try:
            gt_annotations_df = load_gt_annotations(gt_path)
            print(f"GT rows loaded: {len(gt_annotations_df)}")
        except Exception as exc:
            print(f"[WARN] Failed to load GT annotations ({gt_path}): {exc}")

    case_output_dir = OUTPUT_DIR if run_mode == "single" else \
        str(Path(BATCH_OUTPUT_ROOT) / f"baseline_yolo_candidates_{_safe_case_name(case_id)}")

    case_result_df, case_run_info = process_video_baseline(
        video_path=video_path,
        output_dir=case_output_dir,
        sample_fps=SAMPLE_FPS,
        min_sample_fps=MIN_SAMPLE_FPS,
        flow_magnitude_thr=FLOW_MAGNITUDE_THR,
        flow_resize_width=FLOW_RESIZE_WIDTH,
        temporal_window_sec=TEMPORAL_WINDOW_SEC,
        top_k_frames_per_window=TOP_K_FRAMES_PER_WINDOW,
        # max_glare_ratio_for_ocr=MAX_GLARE_RATIO_FOR_OCR,
        # tenengrad_median_factor=TENENGRAD_MEDIAN_FACTOR,
        # min_brightness_for_ocr=MIN_BRIGHTNESS_FOR_OCR,
        # max_brightness_for_ocr=MAX_BRIGHTNESS_FOR_OCR,
        max_frames=MAX_FRAMES,
        gt_df=gt_annotations_df,
        frame_rotation=FRAME_ROTATION_MODE,
    )

    if not case_result_df.empty:
        case_result_df = case_result_df.copy()
        case_result_df["case_id"] = case_id
        case_result_df["video_path"] = video_path
        case_result_df["split"] = "test" if is_test else "unlabeled"
        all_result_parts.append(case_result_df)

    fq_path = case_run_info.get("frame_quality_csv_path")
    if fq_path and Path(fq_path).exists():
        fq_df = pd.read_csv(fq_path)
        fq_df["case_id"] = case_id
        all_frame_quality_parts.append(fq_df)

    ms = case_run_info.get("metrics_summary", {})
    full = ms.get("full_detection_metrics", {})
    proxy = ms.get("proxy_business_metrics", {})
    summary_rows.append({
        "case_id": case_id, "video_path": video_path,
        "split": "test" if is_test else "unlabeled",
        "gt_used": bool(gt_annotations_df is not None and not gt_annotations_df.empty),
        "output_dir": case_run_info.get("output_dir", case_output_dir),
        "frames_processed": ms.get("frames_processed", np.nan),
        "detections_total": ms.get("detections_total", np.nan),
        "precision@0.5": full.get("precision@0.5", np.nan),
        "recall@0.5": full.get("recall@0.5", np.nan),
        "f1@0.5": full.get("f1@0.5", np.nan),
        "ap@0.5": full.get("ap@0.5", np.nan),
        "mean_iou": full.get("mean_iou", np.nan),
        "duplicate_rate_eval": full.get("duplicate_rate_eval", np.nan),
        "candidates_per_frame_mean": proxy.get("candidates_per_frame_mean", np.nan),
        "mean_quality_score": proxy.get("mean_quality_score", np.nan),
        "mean_sharpness": proxy.get("mean_sharpness", np.nan),
        "frame_sharpness_score_mean": proxy.get("frame_sharpness_score_mean", np.nan),
        "fallback_usage_rate": proxy.get("fallback_usage_rate", np.nan),
        "processing_time_sec": proxy.get("processing_time_sec", np.nan),
        "effective_fps": proxy.get("effective_fps", np.nan),
    })
    all_run_infos.append(case_run_info)
    if not case_result_df.empty:
        show_inference_samples(
            case_result_df, video_path, case_id,
            n_samples=8, rotation=FRAME_ROTATION_MODE,
        )

result_df = pd.concat(all_result_parts, ignore_index=True) if all_result_parts else pd.DataFrame()
frame_quality_df_all = pd.concat(all_frame_quality_parts, ignore_index=True) if all_frame_quality_parts else pd.DataFrame()
batch_summary_df = pd.DataFrame(summary_rows)

print("\nBatch summary shape:", batch_summary_df.shape)
display(batch_summary_df)
print("\nCombined result_df shape:", result_df.shape)
display(result_df.head())
print("\nCombined frame_quality_df_all shape:", frame_quality_df_all.shape)
display(frame_quality_df_all.head())

run_info = all_run_infos[-1] if all_run_infos else {}
metrics_df = batch_summary_df


## 13. Visual inspection


In [ ]:
def _read_rgb(path: str) -> np.ndarray | None:
    img = cv2.imread(path)
    if img is None:
        return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def show_image_grid_with_titles(
    paths: list[str],
    titles: list[str] | None,
    title: str,
    cols: int = 4,
) -> None:
    if not paths:
        print(f"No images to display for: {title}")
        return

    rows = int(math.ceil(len(paths) / cols))
    plt.figure(figsize=(4.5 * cols, 3.8 * rows))
    for i, p in enumerate(paths, start=1):
        img = _read_rgb(p)
        plt.subplot(rows, cols, i)
        if img is None:
            plt.text(0.5, 0.5, "Image not found", ha="center", va="center")
            plt.axis("off")
            continue
        plt.imshow(img)
        cap = Path(p).name
        if titles and i - 1 < len(titles):
            cap = titles[i - 1]
        plt.title(cap, fontsize=9)
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def show_preprocess_before_after(frame_paths: list[str], n: int = 6) -> None:
    if not frame_paths:
        print("No frame paths for preprocess preview.")
        return

    sampled = frame_paths[:n]
    for fp in sampled:
        img = cv2.imread(fp)
        if img is None:
            continue
        rot = _apply_rotation(img, FRAME_ROTATION_MODE)
        prep = preprocess_frame_bundle(rot)

        fig, axes = plt.subplots(1, 4, figsize=(18, 4))
        axes[0].imshow(cv2.cvtColor(rot, cv2.COLOR_BGR2RGB))
        axes[0].set_title("Rotated input")
        axes[1].imshow(prep["clahe"], cmap="gray")
        axes[1].set_title("CLAHE")
        axes[2].imshow(prep["denoised"], cmap="gray")
        axes[2].set_title("Denoised")
        axes[3].imshow(prep["sharpen"], cmap="gray")
        axes[3].set_title("Sharpened (OCR input)")
        for ax in axes:
            ax.axis("off")
        fig.suptitle(f"Preprocessing chain: {Path(fp).name}")
        plt.tight_layout()
        plt.show()


def top_paths_by_score(df: pd.DataFrame, path_col: str, score_col: str, top_n: int) -> tuple[list[str], list[str]]:
    if df.empty or path_col not in df.columns or score_col not in df.columns:
        return [], []
    work = df.sort_values(score_col, ascending=False).head(top_n)
    paths = work[path_col].astype(str).tolist()
    titles = [f"{score_col}={v:.4f}" for v in work[score_col].astype(float).tolist()]
    return paths, titles


if result_df.empty:
    print("No candidates were produced. Check INPUT_VIDEO_PATH, SAMPLE_FPS, OCR availability, and fallback settings.")
else:
    print("Step A. Show sampled extracted frames (after video slicing)")
    first_case_output_dir = None
    if "batch_summary_df" in globals() and isinstance(batch_summary_df, pd.DataFrame) and not batch_summary_df.empty:
        first_case_output_dir = str(batch_summary_df.iloc[0]["output_dir"])
    elif "run_info" in globals() and run_info:
        first_case_output_dir = run_info.get("output_dir")

    sampled_frame_paths = []
    if first_case_output_dir:
        frame_dir = Path(first_case_output_dir) / "frames"
        sampled_frame_paths = [str(p) for p in sorted(frame_dir.glob("*.jpg"))[:TOP_N_DEBUG_FRAMES]]

    show_image_grid_with_titles(sampled_frame_paths, None, "Extracted frames (raw from video)", cols=3)

    print("Step B. Preprocessing before/after snapshots")
    show_preprocess_before_after(sampled_frame_paths, n=min(PREPROCESS_PREVIEW_N, len(sampled_frame_paths)))

    print("Step C. Frame ranking by different frame-level scores")
    if "frame_quality_df_all" in globals() and isinstance(frame_quality_df_all, pd.DataFrame) and not frame_quality_df_all.empty:
        p1, t1 = top_paths_by_score(frame_quality_df_all, "rotated_frame_path", "frame_sharpness_score", TOP_N_DEBUG_FRAMES)
        show_image_grid_with_titles(p1, t1, f"Top-{TOP_N_DEBUG_FRAMES} frames by frame_sharpness_score", cols=3)

        p2, t2 = top_paths_by_score(frame_quality_df_all, "rotated_frame_path", "frame_custom_score", TOP_N_DEBUG_FRAMES)
        show_image_grid_with_titles(p2, t2, f"Top-{TOP_N_DEBUG_FRAMES} frames by frame_custom_score", cols=3)
    else:
        print("frame_quality_df_all is empty, skip frame score comparison")

    print("Step D. Candidate debug frames with bbox")
    debug_paths = (
        result_df["debug_frame_path"]
        .dropna()
        .drop_duplicates()
        .head(TOP_N_DEBUG_FRAMES)
        .tolist()
    )
    show_image_grid_with_titles(debug_paths, None, "Debug frames with candidate bboxes", cols=3)

    print("Step E. Top-N crops by different crop-level scores")
    pq, tq = top_paths_by_score(result_df, "crop_path", "quality_score", TOP_N_CROPS)
    show_image_grid_with_titles(pq, tq, f"Top-{TOP_N_CROPS} crops by quality_score", cols=4)

    ps, ts = top_paths_by_score(result_df, "crop_path", "sharpness", TOP_N_CROPS)
    show_image_grid_with_titles(ps, ts, f"Top-{TOP_N_CROPS} crops by sharpness", cols=4)

    pc, tc = top_paths_by_score(result_df, "crop_path", "candidate_score", TOP_N_CROPS)
    show_image_grid_with_titles(pc, tc, f"Top-{TOP_N_CROPS} crops by candidate_score", cols=4)

    print("Step F. Random crop audit (manual suspicious check)")
    sample_n = min(TOP_N_CROPS, len(result_df))
    rand_df = result_df.sample(n=sample_n, random_state=RANDOM_SEED) if sample_n > 0 else pd.DataFrame()
    rand_paths = rand_df["crop_path"].astype(str).tolist() if not rand_df.empty else []
    rand_titles = [
        f"q={r.quality_score:.3f} | sh={r.sharpness:.1f} | cs={r.candidate_score:.2f}"
        for r in rand_df.itertuples(index=False)
    ] if not rand_df.empty else []
    show_image_grid_with_titles(rand_paths, rand_titles, f"Random {sample_n} crops for sanity check", cols=4)

    print("Step G. Best crop selection vs rejected candidates (same frame)")
    if all(col in result_df.columns for col in ["rank_by_quality", "rank_by_sharpness", "rank_by_candidate"]):
        multi_df = result_df.groupby("frame_index").filter(lambda g: len(g) >= 3)
        frame_ids = multi_df["frame_index"].drop_duplicates().head(4).tolist()
        compare_paths: list[str] = []
        compare_titles: list[str] = []

        for fid in frame_ids:
            g = multi_df[multi_df["frame_index"] == fid].copy()
            if g.empty:
                continue

            q_best = g.sort_values("quality_score", ascending=False).iloc[0]
            s_best = g.sort_values("sharpness", ascending=False).iloc[0]
            c_best = g.sort_values("candidate_score", ascending=False).iloc[0]
            rejected = g.sort_values("quality_score", ascending=True).iloc[0]

            entries = [
                (q_best, "BEST quality"),
                (s_best, "BEST sharpness"),
                (c_best, "BEST candidate"),
                (rejected, "REJECT (lowest quality)"),
            ]
            for row, tag in entries:
                compare_paths.append(str(row.crop_path))
                compare_titles.append(
                    f"f={fid} {tag}\nq={row.quality_score:.3f} sh={row.sharpness:.1f} cs={row.candidate_score:.2f}"
                )

        show_image_grid_with_titles(compare_paths, compare_titles, "Selection strategy comparison: keep vs reject", cols=4)
    else:
        print("Rank columns not found, skip selection strategy comparison")

    print("Step H. Future improvements and expected effect")
    improvements = pd.DataFrame(
        [
            {"idea": "Temporal tracking across frames", "expected_effect": "-20%..-40% duplicate_rate_video, more stable best-crop choice"},
            {"idea": "Small detector model (YOLO/RT-DETR) before OCR", "expected_effect": "+10%..+25% recall on partially occluded tags"},
            {"idea": "QR decoding branch before OCR text heuristics", "expected_effect": "+barcode_read_rate and better business matching"},
            {"idea": "Perspective rectification for angled tags", "expected_effect": "+5%..+15% OCR quality on skewed labels"},
            {"idea": "Learned crop scoring head", "expected_effect": "better top-N precision than fixed quality formula"},
        ]
    )
    display(improvements)


## 14. Save artifacts


In [ ]:
artifacts_root = Path("/kaggle/working")
if not artifacts_root.exists():
    artifacts_root = Path(".")

if "all_run_infos" not in globals() or not all_run_infos:
    raise RuntimeError("No run info found. Execute the run cell first.")

# Save combined summary tables
batch_summary_csv = artifacts_root / "batch_metrics_summary.csv"
batch_summary_json = artifacts_root / "batch_metrics_summary.json"

if "batch_summary_df" in globals() and isinstance(batch_summary_df, pd.DataFrame):
    batch_summary_df.to_csv(batch_summary_csv, index=False)
    print(f"Saved: {batch_summary_csv}")

    with batch_summary_json.open("w", encoding="utf-8") as f:
        json.dump(batch_summary_df.to_dict(orient="records"), f, ensure_ascii=False, indent=2)
    print(f"Saved: {batch_summary_json}")


if "frame_quality_df_all" in globals() and isinstance(frame_quality_df_all, pd.DataFrame) and not frame_quality_df_all.empty:
    frame_quality_all_csv = artifacts_root / "frame_quality_all.csv"
    frame_quality_df_all.to_csv(frame_quality_all_csv, index=False)
    print(f"Saved: {frame_quality_all_csv}")

if "result_df" in globals() and isinstance(result_df, pd.DataFrame) and not result_df.empty:
    combined_debug_csv = artifacts_root / "debug_candidates_all.csv"
    result_df.to_csv(combined_debug_csv, index=False)
    print(f"Saved: {combined_debug_csv}")

zip_paths = []
for info in all_run_infos:
    out_dir = Path(info["output_dir"])
    if not out_dir.exists():
        continue

    src_debug_csv = Path(info["debug_csv_path"])
    if src_debug_csv.exists():
        target_name = f"{out_dir.name}_debug_candidates.csv"
        shutil.copy2(src_debug_csv, artifacts_root / target_name)


src_frame_quality_csv = Path(info.get("frame_quality_csv_path", ""))
if src_frame_quality_csv.exists():
    shutil.copy2(src_frame_quality_csv, artifacts_root / f"{out_dir.name}_frame_quality_debug.csv")

    src_metrics_csv = Path(info["metrics_csv_path"])
    src_metrics_json = Path(info["metrics_json_path"])
    if src_metrics_csv.exists():
        shutil.copy2(src_metrics_csv, artifacts_root / f"{out_dir.name}_metrics_summary.csv")
    if src_metrics_json.exists():
        shutil.copy2(src_metrics_json, artifacts_root / f"{out_dir.name}_metrics_summary.json")

    zip_base = artifacts_root / out_dir.name
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(out_dir))
    zip_paths.append(zip_path)

# Zip all per-case output directories together
if run_mode == "batch":
    batch_root = Path(BATCH_OUTPUT_ROOT)
    if batch_root.exists():
        batch_zip = shutil.make_archive(str(artifacts_root / "baseline_ocr_candidates_batch_all"), "zip", root_dir=str(batch_root))
        print(f"Batch zip created: {batch_zip}")

print("Per-case zip archives:")
for z in zip_paths:
    print("  -", z)
